# Анализ и прогнозирование временных рядов методами искусственного интеллекта

## **Практическая работа 2. Поиск по образцу.**

### **Часть 0.** Подготовка окружения и ноутбука

In [1]:
import os
from pathlib import Path

practice_dir_path = Path(os.getcwd()).absolute()
print(practice_dir_path)

os.chdir(practice_dir_path)

c:\Users\admin\Projects\labs\5\3\tsc\practice\02 Similarity search


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import numpy as np
import pandas as pd
import math
import timeit
import random
import mass_ts as mts
from IPython.display import display

from modules.distance_profile import brute_force
from modules.prediction import *
from modules.bestmatch import *
from modules.utils import *
from modules.plots import *
from modules.experiments import *

c:\Users\admin\Projects\labs\5\3\tsc\.venv\Lib\site-packages\mass_ts\_mass_ts.py:17: UserWarning: GPU support will not work. You must pip install mass-ts[gpu].
  warnings.warn(


### **Часть 1.** Поиск по образцу на основе вычисленного профиля расстояния.

**Задача поиска по образцу (subsequence matching)** заключается в нахождении $topK$ наиболее похожих подпоследовательностей временного ряда $T$ длины $n$ на заданный пользователем существенно более короткий временной ряд $Q$ длины $m$, называемый запросом, в смысле некоторой функции расстояния, $m \ll n$. При этом среди найденных подпоследовательностей не должно быть тривиальных совпадений.

Две подпоследовательности $T_{i,m}$ и $T_{j,m}$ временного ряда $T$ являются **тривиальными совпадениями (trivial matches)**, если они пересекаются:
\begin{equation}
|i-j| \leqslant m.
\end{equation}

В общем случае условие пересечения записывается как $|i-j| \leqslant \xi m$, где задаваемый экспертом вещественный параметр $\xi$ $(0 < \xi \leqslant 1)$ имеет типичные значения 0.25, 0.5 или 1.

Одним из вариантов решения данной задачи является вычисление профиля расстояния. Под **профилем расстояния** $DistProfile\in \mathbb{R}^{n-m+1}$ понимается вектор, содержащий расстояния между подпоследовательностями временного ряда $T \in \mathbb{R}^n$ и запросом $Q \in \mathbb{R}^m$, вычисленные с помощью некоторой неотрицательной симметричной функции расстояния $dist(\cdot,\cdot)$:  
\begin{equation}
DistProfile(i) = dist(Q, T_{i,m}), \quad 1 \leqslant i \leqslant n-m+1.
\end{equation}

На основе вычисленного профиля расстояния в качестве $topK$ похожих подпоследовательностей ряда берутся те, которые имеют наименьшие расстояния до запроса:
\begin{equation}
C_{match} = \{T_{i,m}^k\}_{k=1}^{topK},\; где \; T_{i,m}^k \in T, \; i = argsort(DistProfile)(k), \; 1 \leqslant i \leqslant n-m+1.
\end{equation}

В части 1 практической работы 2 вы рассмотрите несколько алгоритмов вычисления профиля расстояния на примере алгоритмов грубой силы и MASS, а также примените результаты их выполнения для решения задачи поиска по образцу. В таблице ниже представлено их тезисное описание.

| <h5> **Алгоритм** </h5> | <h5> **Описание** </h5> | <h5> **Вычислительная <br> сложность** </h5> |
|--------------|------------------------------------------------------------------|:-----------------------------:|
| <p>Brute Force</p>  | <ul><li>Наивный алгоритм</li><li>Z-нормализация запроса и подпоследовательностей ряда по стандартным формулам</li><li>Полное вычисление расстояний между запросом и подпоследовательностями ряда</li></ul> |            <p>$O(mn)$</p>            |
| <p>MASS 1</p>       | <ul><li>Z-нормализация &#171;на лету&#187;</li><li>Применение свертки для вычисления скалярных произведений <br> между запросом и подпоследовательностями ряда</li><li>Выполнение свертки с помощью быстрого преобразования Фурье</li><li>Дополнение справа нулями запроса и временного ряда до удвоенной длины ряда</li><li>Реверс запроса</li></ul> |          <p>$O(n\log{n})$</p>          |
| <p>MASS 2</p>       | <ul><li>MASS 1 – это алгоритм, на котором основан MASS 2</li><li>Вычисление половины свертки</li><li>Дополнение нулями справа только запроса до длины временного ряда</li></ul> |          <p>$O(n\log{n})$</p>          |
| <font size="3">MASS 3</font>       | <ul><li>MASS 2 – это алгоритм, на котором основан MASS 3</li><li>Посегментная обработка временного ряда</li><li>Длина каждого сегмента (кроме, возможно, последнего) – степень двойки</li><li>Сегменты перекрываются на $m-1$ элементов</li></ul> |             <p>$O(\frac{n-k}{k-m}k\log{k})$, <br> где $k$ – длина сегмента </p>           |

#### **Задача 1.**

В данном задании вам предстоит определить, имеет ли пациент заболевание сердца по снятой записи ЭКГ или нет. Решать данную задачу будем с помощью алгоритмов поиска по образцу.

Сначала выполните считывание временного ряда и образца поиска из файлов *ECG.csv* и *ECG_query.csv* соответственно из директории *./datasets/part1*. Временной ряд представляет собой показания ЭКГ пациента, образец поиска – фрагмент ЭКГ, обозначающий некоторое кардиологическое заболевание.

In [4]:
ts_url = './datasets/part1/ECG.csv'
query_url = './datasets/part1/ECG_query.csv'

ts = read_ts(ts_url).reshape(-1)
query = read_ts(query_url).reshape(-1)

c:\Users\admin\Projects\labs\5\3\tsc\practice\02 Similarity search\modules\utils.py:20: FutureWarning:

The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead

c:\Users\admin\Projects\labs\5\3\tsc\practice\02 Similarity search\modules\utils.py:20: FutureWarning:

The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead



Далее выполните визулизацию временного ряда и образца поиска с помощью функции `plot_bestmatch_data()` из модуля *plots.py*.

In [5]:
plot_bestmatch_data(ts, query)

Реализуйте алгоритм грубой силы (brute force), заполнив одноименную функцию в модуле *distance_profile.py* недостающим кодом. Для этого воспользуйтесь псевдокодом алгоритма, который представлен ниже. После того как вычислен профиль расстояния, найдите $topK$ похожих подпоследовательностей на запрос с помощью функции `topK_match()` из *bestmatch.py*. Для исключения попадания тривиальных совпадений зададим $\xi = 0.5$ Далее параметр $\xi$ будем обозначать в коде как $excl\_zone\_frac$, а величину пересечения как $excl\_zone$, $excl\_zone = \lceil excl\_zone\_frac \cdot m \rceil$,

<center><img src="./img/brute_force.png?raw=true" width="400"></center>

In [6]:
topK = 2
excl_zone_frac = 0.5
excl_zone = math.ceil(len(query) * excl_zone_frac)
is_normalize = True

dist_profile = brute_force(ts, query, is_normalize=is_normalize)
matches = topK_match(dist_profile, topK=topK, excl_zone=excl_zone)

naive_bestmatch_results = {
    "dist_profile": dist_profile,
    "matches": matches,
    "topK": topK,
    "excl_zone": excl_zone,
}
naive_bestmatch_results

{'dist_profile': array([22.39707072, 22.23816137, 22.16184162, ..., 26.54153207,
        26.7563139 , 26.96410961], shape=(2580,)),
 'matches': {'indices': [np.int64(1215), np.int64(193)],
  'distances': [np.float64(5.01641797172885), np.float64(18.030850173768773)]},
 'topK': 2,
 'excl_zone': 211}

Напишите функцию `plot_bestmatch_results()` в модуле *plots.py* для визуализации найденных $topK$ похожих подпоследовательностей временного ряда на образец поиска. За основу возьмите функцию `plot_bestmatch_data()` и добавьте отображение найденных подпоследовательностей, выделив их тем же цветом, что и образец поиска.

In [7]:
plot_bestmatch_results(ts, query, naive_bestmatch_results)

> Определите по полученным результатам, имеет ли человек сердечное заболевание или нет.

**Да**, так как есть очень схожее совпадние с искомым фрагментом ЭКГ

#### **Задача 2.**

Проделайте такие же шаги для поиска по образцу, как и в задаче 1, но теперь вычислите профиль расстояния с помощью одной из выбранных вами версий алгоритма MASS. Для этого используйте стороннюю библиотеку *mass-ts*. Со списком всех функций, которые предоставляет данная библиотека, и их описанием вы можете ознакомиться в [репозитории библиотеки GitHub](https://github.com/matrix-profile-foundation/mass-ts).

Обратите внимание, что MASS 2 и MASS 3 возвращают профиль расстояния в виде вектора комплексных чисел. Для дальнейшей работы с профилем расстояния используйте только вещественные части комплексных чисел.     

In [8]:
dist_profile = mts.mass(ts, query)
dist_profile_real = dist_profile.real

In [9]:
matches = np.argsort(dist_profile_real[excl_zone:])[:topK] + excl_zone

mass_bestmatch_results = {
    "dist_profile": dist_profile_real,
    "matches": {
        "indices": matches.tolist(),
        "distances": [dist_profile[idx] for idx in matches]
    },
    "topK": topK,
    "excl_zone": excl_zone,
}

mass_bestmatch_results

{'dist_profile': array([22.23816137, 22.16184162, 22.18146561, ..., 26.54153207,
        26.7563139 , 26.96410961], shape=(2579,)),
 'matches': {'indices': [1214, 1215],
  'distances': [np.float64(5.0164179717288615),
   np.float64(5.336777989950037)]},
 'topK': 2,
 'excl_zone': 211}

In [10]:
plot_bestmatch_results(ts, query, mass_bestmatch_results)

#### **Задача 3.**

Проведите две серии экспериментов по сравнению быстродействия алгоритмов грубой силы и трех версий MASS:
1. на фиксированной длине запроса $m$ при изменяемой длине временного ряда $n$;
2. на фиксированной длине временного ряда $n$ при изменяемой длине запроса $m$.

Все необходимые для проведения экспериментов функции находятся в модуле *experiments.py*.

Сначала сгенерируйте по аналогии временные ряды и запросы поиска различных длин, как это было сделано в практической работе 1. Далее измерьте время выполнения алгоритмов при заданных входных параметрах и данных с помощью функции `run_experiment()`. Полученные результаты (время выполнения) каждого эксперимента отобразите на линейном графике, воспользовавшись функцией `visualize_plot_times()`.

Также вычислите ускорение с помощью функции `calculate_speedup()`, показывающее, во сколько раз алгоритм MASS превосходит по времени выполнения алгоритма грубой силы, по следующей формуле:
\begin{equation}
speedup = \frac{t_{BF}}{t_{MASS}},
\end{equation}
где $t_{BF}$ и $t_{MASS}$ — время работы алгоритма грубой силы и MASS соответственно.

Полученные ускорения оформите в виде таблицы, для построения используйте функцию `visualize_table_speedup()`.

##### Эксперимент 1

In [11]:
sm_algorithms = ['brute_force', 'mass', 'mass2', 'mass3']
sm_algorithms_params = {
    'brute_force': None,
    'mass': None,
    'mass2': None,
    'mass3': {'segment_len': 2048},
}

sm1_n_list = [2**15, 2**16, 2**17, 2**18, 2**19, 2**20] # lengths of time series
sm1_m = 128 # length of query

sm1_exp_params = {
    'varying': {'n': sm1_n_list},
    'fixed': {'m': sm1_m}
}

sm1_exp_data = {
    'ts': dict.fromkeys(map(str, sm1_n_list), []),
    'query': {str(sm1_m): []}
}

for sm1_n in sm1_n_list:
    ts_array = np.random.rand(sm1_n)
    query_array = np.random.rand(sm1_m)
    sm1_exp_data['ts'][str(sm1_n)] = ts_array
    sm1_exp_data['query'][str(sm1_m)] = query_array  # same query for all n


sm1_exp_results = {}
for sm_algorithm in sm_algorithms:
    sm1_exp_results[sm_algorithm] = run_experiment(
        algorithm=sm_algorithm,
        task='distance_profile',
        data=sm1_exp_data,
        exp_params=sm1_exp_params,
        alg_params=sm_algorithms_params[sm_algorithm],
    )

In [12]:
visualize_plot_times(np.array(list(sm1_exp_results.values())), np.array(sm_algorithms), sm1_exp_params)

In [13]:
# visualize table with speedup

sm1_t_bf = sm1_exp_results['brute_force']

sm1_tab_index = sm_algorithms[1:]
sm1_tab_columns = [f"n = {sm1_n}" for sm1_n in sm1_n_list]
sm1_tab_title = "Speedup MASS relative to the brute force <br> (variable time series length, fixed query length)"

sm1_speedups = []
for sm_alghorithm in sm_algorithms[1:]:
    sm1_t_mass = sm1_exp_results[sm_alghorithm]
    sm1_sp = calculate_speedup(sm1_t_bf, sm1_t_mass)
    sm1_speedups.append(sm1_sp)

visualize_table_speedup(np.array(sm1_speedups), sm1_tab_index, sm1_tab_columns, sm1_tab_title)

,n = 32768,n = 65536,n = 131072,n = 262144,n = 524288,n = 1048576
mass,144.669255,122.672949,123.490525,95.882734,88.744412,87.026091
mass2,77.150542,63.037930,62.340810,53.947262,45.621903,44.057709
mass3,76.707411,56.347164,56.470867,43.895889,33.497580,23.166797


##### Эксперимент 2

In [14]:
sm2_m_list = [2**5, 2**6, 2**7, 2**8, 2**9, 2**10] # lengths of queries
sm2_n = 2**15 # length of time series

sm2_exp_params = {
    'varying': {'m': sm2_m_list},
    'fixed': {'n': sm2_n}
}

sm2_exp_data = {
    'ts': {str(sm2_n): []},
    'query': dict.fromkeys(map(str, sm2_m_list), [])
}


for sm2_m in sm2_m_list:
    sm2_ts_array = np.random.rand(sm2_n)
    sm2_query_array = np.random.rand(sm2_m)
    sm2_exp_data['ts'][str(sm2_n)] = sm2_ts_array
    sm2_exp_data['query'][str(sm2_m)] = sm2_query_array


sm2_exp_results = {}
for sm_algorithm in sm_algorithms:
    sm2_exp_results[sm_algorithm] = run_experiment(
        algorithm=sm_algorithm,
        task='distance_profile',
        data=sm2_exp_data,
        exp_params=sm2_exp_params,
        alg_params=sm_algorithms_params[sm_algorithm],
    )

In [15]:
visualize_plot_times(np.array(list(sm2_exp_results.values())), np.array(sm_algorithms), sm2_exp_params)

In [16]:
sm2_t_bf = sm2_exp_results['brute_force']

sm2_tab_index = sm_algorithms[1:]
sm2_tab_columns = [f"m = {sm2_m}" for sm2_m in sm2_m_list]
sm2_tab_title = "Speedup MASS relative to the brute force <br> (variable query length, fixed time series length)"

sm2_speedups = []
for sm_algorithm in sm_algorithms[1:]:
    sm2_t_mass = sm2_exp_results[sm_algorithm]
    sp = calculate_speedup(sm2_t_bf, sm2_t_mass)
    sm2_speedups.append(sp)

visualize_table_speedup(np.array(sm2_speedups), sm2_tab_index, sm2_tab_columns, sm2_tab_title)

,m = 32,m = 64,m = 128,m = 256,m = 512,m = 1024
mass,107.871402,104.041048,115.936951,117.550159,110.246865,116.149572
mass2,120.055843,77.181212,54.086713,31.637292,15.327057,8.403629
mass3,144.107303,86.414910,54.175641,34.575952,16.640440,8.641501


##### Выводы

> Проанализируйте и изложите содержательный смысл полученных результатов.

Таким образом, проведенные эксперименты убедительно доказывают, что все реализации алгоритма MASS на порядки превосходят по производительности метод грубой силы. Это объясняется их более эффективной вычислительной сложностью O(n·log n) на основе быстрого преобразования Фурье по сравнению с O(n·m) у brute_force. Независимо от длины временного ряда или запроса, использование любой версии MASS обеспечивает ускорение вычислений в десятки, а то и в сотни раз, что делает их индустриальным стандартом для решения задачи поиска схожих подпоследовательностей.

Ключевое различие между версиями MASS заключается в их реакции на изменение длины запроса m. Классическая реализация mass демонстрирует уникальное свойство: её время работы практически не зависит от длины m, что делает её универсальным и самым надёжным выбором, особенно для поиска длинных паттернов. В то же время, производительность mass2 и mass3 линейно деградирует с ростом m. Эти версии показывают преимущество только для очень коротких запросов, но быстро уступают mass в эффективности, что говорит об их узкой специализации, а не об универсальном применении.

### **Часть 2.** Ускорение вычисления DTW меры техникой ограничения полосы Сако—Чиба. Наивный алгоритм поиска по образцу на основе DTW меры.

#### **Задача 4.**

Поскольку DTW мера имеет квадратичную вычислительную сложность от
длины временного ряда $O(n^2)$, то в данном задании вам предстоит реализовать технику ограничения полосы Сако–Чиба. Данная техника не позволяет отклоняться пути трансформации более чем на $r$ ячеек от диагонали матрицы трансформации и тем самым сокращает вычисление меры до $O(rn)$.

Добавьте в функцию `DTW_distance()` из модуля *metrics.py* возможность ограничения полосы Сако—Чиба. Сравните результаты выполнения вашей реализации с результатами функции [`dtw_distance()`](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.distances.dtw_distance.html) из библиотеки *sktime*, задав различные значения параметра $r$ (например, от 0 до 1 с шагом 0.05).

**Мера DTW с ограничением полосы Сако–Чиба** вычисляется следующим образом:
\begin{equation}
\text{DTW}(T_1, T_2) = d(n,n),
\\ d(i,j) = (t_{1,i} - t_{2,j})^2 + \min \left\{
	\begin{array}{l l}
	d(i-1,j), \\
	d(i,j-1), \\
	d(i-1,j-1),
	\end{array}
	\right.
\\ d(0,0)=0, \quad d(i,0)=d(0,j)=\infty, \quad  1 \leqslant i,j \leqslant n;
\\ 0 \leqslant r \leqslant n-1, \quad j-r \leqslant i \leqslant j+r,
\\ d(i,j) = \infty, \quad j+r < i < j-r.
\end{equation}

In [17]:
def test_distances(dist1: float, dist2: float) -> None:
    """
    Check whether your distance function is implemented correctly

    Parameters
    ----------
    dist1 : distance between two time series calculated by sktime
    dist2 : distance between two time series calculated by your function
    """

    np.testing.assert_equal(round(dist1, 5), round(dist2, 5), 'Distances are not equal')

In [22]:
from sktime.distances import dtw_distance
from modules.metrics import DTW_distance

ts1 = np.random.randn(128)
ts2 = np.random.randn(128)

r_values = np.arange(0.0, 1.05, 0.05)

for r in r_values:
    dist1 = dtw_distance(ts1, ts2, window=r)
    dist2 = DTW_distance(ts1, ts2, r)
    test_distances(dist1, dist2)

#### **Задача 5.**

Реализуйте самостоятельно наивный алгоритм поиска $topK$ подпоследовательностей временного ряда, похожих на образец поиска в смысле меры DTW. Для этого уже подготовлен шаблон класса `NaiveBestMatchFinder` в модуле *bestmatch.py*. Напишите метод `perform()`, выполняющий обнаружение подпоследовательностей ряда, похожих на образец поиска. Метод должен избегать попадания в результирующее множество пересекающихся подпоследовательностей. Для этого используйте функцию `topK_match()`. При реализации опирайтесь на псевдокод наивного алгоритма поиска, который представлен ниже.

<center><img src="./img/naive_algorithm.png" width="550"></center>

Для этой задачи используйте временной ряд и запрос ЭКГ из части 1. Если они не загружены, то выполните их считывание из соответствующих файлов.

In [23]:
topK = 2
r = 0.01
excl_zone_frac = 0.5
is_normalize = True

naive_finder = NaiveBestMatchFinder(
    excl_zone_frac=excl_zone_frac,
    topK=topK,
    is_normalize=is_normalize,
    r=r
)

naive_bestmatch_results = naive_finder.perform(ts, query)

Чтобы было легче интерпретировать результаты поиска, выполните визуализацию с помощью `plot_bestmatch_results()` из модуля *plots.py*.

In [24]:
plot_bestmatch_results(ts, query, naive_bestmatch_results)

#### **Задача 6**.

Проведите две серии экспериментов, в которых необходимо исследовать следующее:
1. влияние длины запроса $m$ и ширины полосы Сако—Чиба $r$ на время выполнения наивного алгоритма при фиксированной длине ряда $n$;
2. влияние длины ряда $n$ и ширины полосы Сако—Чиба $r$ на время выполнения наивного алгоритма при фиксированной длине запроса $m$.

Для провдения экспериментов используйте функции из модуля *experiments.py*.
Эксперименты проведите на синтетических временных рядах и запросах указанных длин. Полученные результаты каждого эксперимента отобразите на линейном графике.

##### Эксперимент 1

In [25]:
naive_algorithm = 'naive'
naive_algorithm_params = {
    'topK': 3,
    'excl_zone_frac': 1,
    'normalize': True,
}

naive1_n_list = [2**10, 2**11, 2**12, 2**13, 2**14, 2**15] # lengths of time series
naive1_r_list = np.round(np.arange(0, 0.6, 0.1), 2).tolist() # sizes of warping window
naive1_m = 2**6 # length of query

naive1_exp_params = {
    'varying': {'n': naive1_n_list,
                'r': naive1_r_list},
    'fixed': {'m': naive1_m}
}
naive1_exp_data = {
    'ts': dict.fromkeys(map(str, naive1_n_list), []),
    'query': {str(naive1_m): []}
}

for naive1_n in naive1_n_list:
    naive1_ts_array = np.random.rand(naive1_n)
    naive1_query_array = np.random.rand(naive1_m)
    naive1_exp_data['ts'][str(naive1_n)] = naive1_ts_array
    naive1_exp_data['query'][str(naive1_m)] = naive1_query_array

naive1_exp_results = run_experiment(
    algorithm=naive_algorithm,
    task='best_match',
    data=naive1_exp_data,
    exp_params=naive1_exp_params,
    alg_params=naive_algorithm_params,
)

In [26]:
visualize_plot_times(naive1_exp_results, np.array(naive1_r_list), naive1_exp_params)

##### Эксперимент 2

In [27]:
naive2_m_list = [2**4, 2**5, 48, 2**6, 2**7, 2**8] # lengths of query
naive2_r_list = np.round(np.arange(0, 0.6, 0.1), 2).tolist() # sizes of warping window
naive2_n = 2**12 # length of time series

naive2_exp_params = {
    'varying': {'m': naive2_m_list,
                'r': naive2_r_list},
    'fixed': {'n': naive2_n}
}
naive2_exp_data = {
    'ts': {str(naive2_n): []},
    'query': dict.fromkeys(map(str, naive2_m_list), []),
}


ts_array = np.random.rand(naive2_n)
naive2_exp_data['ts'][str(naive2_n)] = ts_array

for naive2_m in naive2_m_list:
    naive2_query_array = np.random.rand(naive2_m)
    naive2_exp_data['query'][str(naive2_m)] = naive2_query_array

naive2_exp_results = run_experiment(
    algorithm=naive_algorithm,
    task='best_match',
    data=naive2_exp_data,
    exp_params=naive2_exp_params,
    alg_params=naive_algorithm_params,
)

In [28]:
visualize_plot_times(naive2_exp_results, np.array(naive2_r_list), naive2_exp_params)

##### Выводы

> Проанализируйте и изложите содержательный смысл полученных результатов.

Анализ показывает, что производительность наивного алгоритма демонстрирует линейную зависимость от длины ряда $n$, но гораздо более критичную квадратичную зависимость от длины запроса $m$. Это связано с тем, что сложность базовой операции — вычисления DTW-расстояния — пропорциональна $m^2$, и эта операция повторяется $n$ раз. Ширина полосы Сако—Чиба $r$ выступает дополнительным мультипликативным фактором, увеличивающим общее время работы. Таким образом, длина запроса $m$ является основным фактором, ограничивающим производительность алгоритма, делая его непрактичным для длинных запросов даже при умеренной длине временного ряда.

### **Часть 3.** Алгоритм UCR-DTW.

Третья часть практической работы посвящена алгоритму UCR-DTW, который использует нижние границы схожести $\text{LB}_{\text{Kim}}\text{FL}$, $\text{LB}_{\text{Keogh}}\text{EQ}$ и $\text{LB}_{\text{Keogh}}\text{EC}$, применяющиеся каскадным образом. Псевдокод алгоритма UCR-DTW представлен ниже.

<center><img src="./img/ucr_dtw.png?raw=true" width="650"></center>

**Нижняя граница схожести (lower bound, LB)** представляет собой функцию, вычислительная сложность которой меньше вычислительной сложности меры DTW. Нижняя граница используется для отбрасывания кандидатов (подпоследовательностей временного ряда), заведомо не похожих на запрос, без вычисления меры DTW.

Нижние границы между кандидатом $C$ и запросом $Q$ длины $n$, применяемые в UCR-DTW, вычисляются следующим образом:

* **Нижняя граница $\text{LB}_{\text{Kim}}\text{FL}$** определяется как сумма квадратов разностей между первыми и последними точками запроса $Q$ и подпоследовательности $C$:
    $$
    \begin{equation}
        \text{LB}_{\text{Kim}}\text{FL}(Q, C) = (q_1 - c_1)^2 + (q_n - c_n)^2.
    \end{equation}
    $$
* **Нижняя граница $\text{LB}_{\text{Keogh}}\text{EQ}$** показывает расстояние между верхней или нижней оболочкой $U$ и $L$, построенными вокруг запроса $Q$, и кандидатом $C$:
    $$
    \begin{equation}
        \text{LB}_{\text{Keogh}}\text{EQ}(Q,C) = \sum_{i=1}^n{\left\{
                \begin{array}{cl}
                (c_i - u_i)^2, & \text{if} \; c_i > u_i \\
                (c_i - l_i)^2, & \text{if} \; c_i < l_i\\
                0, & \text{otherwise}.
                \end{array}
                \right.}
    \end{equation}
    $$
    Нижней и верхней оболочкой (lower and upper envelope) запроса $Q$ называют соответственно последовательности $L = (l_1,..., l_n)$ и $U = (u_1,..., u_n)$, вычисляемые как минимумы и максимумы запроса в скользящем окне заданной длины $r$ ($1 < r < m$):
    $$
    \begin{equation}
        u_i = \max_{\max(1,i-r) \leqslant k \leqslant \min(m, i+r)} q_{k}, \\
        l_i = \min_{\max(1,i-r) \leqslant k \leqslant \min(m, i+r)} q_{k},
    \end{equation}
    $$
    где $r$ – ширина полосы Сако–Чиба.
* **Нижняя граница $\text{LB}_{\text{Keogh}}\text{EC}$** представляет собой расстояние между запросом $Q$ и оболочкой кандидата $C$, т.е. является реверс-версией нижней границы $\text{LB}_{\text{Keogh}}\text{EQ}$:
    $$
    \begin{equation}
        \text{LB}_{\text{Keogh}}\text{EC}(Q,C) = \text{LB}_{\text{Keogh}}\text{EQ}(C, Q).
    \end{equation}
    $$

#### **Задача 7.**

Реализуйте технику каскадного применения нижних границ и сами нижние границы,  заполнив пропуски в классе `UCR_DTW`, в модуле *bestmatch.py*.
Выполните алгоритм UCR-DTW на данных ЭКГ из предыдущих частей, задав такие же значения входных параметров, что и для наивного алгоритма из части 2, и визуализируйте результаты. Убедитесь, что результаты UCR-DTW совпадают с результатами наивного алгоритма.

In [29]:
top_k = 2
r = 0.01
excl_zone_frac = 1
is_normalize = True

ucr_dtw_finder = UCR_DTW(
    excl_zone_frac=excl_zone_frac,
    topK=top_k,
    is_normalize=is_normalize,
    r=r
)

ucr_dtw_results = ucr_dtw_finder.perform(ts, query)
ucr_dtw_stats = ucr_dtw_finder.get_statistics()

Визуализируйте количество неотброшенных и отброшенных каждой нижней границей подпоследовательностей временного ряда в виде круговой диаграммы с помощью функции `pie_chart()` из модуля *plots.py*.

In [30]:
labels = np.array([
    'Неотброшенные', 
    'Отброшенные LB_Kim', 
    'Отброшенные LB_KeoghEQ', 
    'Отброшенные LB_KeoghEC'
])
values = np.array([
    ucr_dtw_stats['not_pruned_num'], 
    ucr_dtw_stats['lb_Kim_num'], 
    ucr_dtw_stats['lb_KeoghQC_num'], 
    ucr_dtw_stats['lb_KeoghCQ_num']
])

pie_chart(
    labels=labels, 
    values=values, 
)

#### **Задача 8.**

Проведите эксперименты, аналогичные тем, которые выполнялись для исследования эффективности наивного алгоритма в задаче 6. Постройте графики и вычислите ускорение алгоритма UCR-DTW относительно наивного алгоритма. Для справедливого сравнения алгоритмов используйте сгенерированные временные ряды и запросы из задачи 6.

##### Эксперимент 1

In [31]:
ucr_algorithm = 'ucr-dtw'
ucr_algorithm_params = {
    'topK': 3,
    'excl_zone_frac': 1,
    'normalize': True,
}

ucr1_exp_results = run_experiment(
    algorithm=ucr_algorithm,
    task='best_match',
    data=naive1_exp_data,
    exp_params=naive1_exp_params,
    alg_params=ucr_algorithm_params,
)

In [32]:
visualize_plot_times(ucr1_exp_results, np.array(naive1_r_list), naive1_exp_params)

In [33]:
ucr1_tab_index = [f"n = {ucr1_n}" for ucr1_n in naive1_n_list]
ucr1_tab_columns = [f"r = {ucr1_r}" for ucr1_r in naive1_r_list]
ucr1_tab_title = "Speedup UCR-DTW relative to the naive algorithm <br> (variable time series length and warping path size, fixed query length)"

ucr1_speedups = naive1_exp_results / ucr1_exp_results

visualize_table_speedup(ucr1_speedups, ucr1_tab_index, ucr1_tab_columns, ucr1_tab_title)

,r = 0.0,r = 0.1,r = 0.2,r = 0.3,r = 0.4,r = 0.5
n = 1024,0.197414,0.159554,0.102465,0.102756,0.096241,0.115315
n = 2048,0.358408,0.344158,0.333326,0.340171,0.320927,0.329801
n = 4096,0.448559,0.438884,0.426136,0.489100,0.623775,0.647525
n = 8192,0.714935,0.710314,0.617668,0.756878,0.760009,0.705215
n = 16384,0.818707,0.743669,0.702378,0.770233,0.772159,0.730537
n = 32768,0.740053,0.768753,0.825087,0.761115,0.758409,0.771703


##### Эксперимент 2

In [34]:
ucr2_exp_results = run_experiment(
    algorithm=ucr_algorithm,
    task='best_match',
    data=naive2_exp_data,
    exp_params=naive2_exp_params,
    alg_params=ucr_algorithm_params,
)

In [35]:
visualize_plot_times(ucr2_exp_results, np.array(naive2_r_list), naive2_exp_params)

In [36]:
ucr2_tab_index = [f"m = {naive2_m}" for naive2_m in naive2_m_list]
ucr2_tab_columns = [f"r = {naive2_r}" for naive2_r in naive2_r_list]
ucr2_tab_title = "Speedup UCR-DTW relative to the naive algorithm <br> (variable query length and warping path size, fixed time series length)"

ucr2_speedups = naive2_exp_results / ucr2_exp_results

visualize_table_speedup(ucr2_speedups, ucr2_tab_index, ucr2_tab_columns, ucr2_tab_title)

,r = 0.0,r = 0.1,r = 0.2,r = 0.3,r = 0.4,r = 0.5
m = 16,0.245442,0.178328,0.152423,0.138836,0.121384,0.096337
m = 32,0.292247,0.318700,0.372547,0.459085,0.608276,0.741365
m = 48,0.508593,0.486737,0.536085,0.591607,0.726344,0.949814
m = 64,0.816837,0.681151,0.831545,0.834778,0.788673,0.874287
m = 128,0.759326,0.589286,0.673402,0.760826,0.831669,0.859290
m = 256,0.774923,0.634686,0.729427,0.755758,0.845856,0.910012


##### Выводы

> Проанализируйте и изложите содержательный смысл полученных результатов.

Эксперимент установил, что оптимизированный алгоритм UCR-DTW продемонстрировал худший результат по сравнению с наивной реализацией, что обусловлено случайным характером использованных временных рядов. Эффективность UCR-DTW, основанная на механизме раннего отсева бесперспективных кандидатов, не проявляется на данных без выраженной структуры, в результате чего вычислительные затраты на построение оптимизаций не компенсируются сокращением числа полных вычислений DTW. Таким образом, производительность алгоритмов зависит от контекста, и подчеркивают важность использования репрезентативных наборов данных.

#### **Задача 9.**

В данном задании вам предстоит определелить, какую функцию расстояния ED или DTW лучше всего использовать на практике для поиска наиболее похожих подпоследовательностей временного ряда на запрос. Чтобы это сделать, рассмотрим две задачи из различных предметных областей, решить которые предлагается с помощью алгоритмов поиска по образцу.

Начнем с **первой задачи из области физиологии человека**. На человеке сначала закрепляют множество акселерометров и гироскопов, после чего он выполняет в помещении последовательность заранее определенных действий (активностей) в течение некоторого промежутка времени. Примерами таких активностей являются открывание/закрывание дверей, включение/выключение света, питье из чашки стоя/сидя и др. В то время как человек выполняет эти активности, датчики фиксируют его скорость и направление наклона тела. Задача дата-сайентиста заключается в том, что необходимо распознать в снятых данных все активности, которые выполнял человек.  

Данную задачу упростим, и будем выполнять поиск только одной активности во временном ряде при условии, что у нас имеется образец этой активности. В качестве данных возьмем временной ряд показаний гироскопа, закрепленного на правом запятье руки человека, из набора данных [Opportunity](https://archive.ics.uci.edu/dataset/226/opportunity+activity+recognition) и образец искомой активности, питья из чашки стоя. Данный ряд соответствует примерно 14-минутной записи.

Загрузите временной ряд и образец поиска из директории `./datasets/part3/Opportunity` в ноутбук.    

In [37]:
ts_opp_url = './datasets/part3/Opportunity/ts.csv'
query_opp_url = './datasets/part3/Opportunity/query.csv'

ts_opp = read_ts(ts_opp_url).reshape(-1)
query_opp = read_ts(query_opp_url).reshape(-1)

c:\Users\admin\Projects\labs\5\3\tsc\practice\02 Similarity search\modules\utils.py:20: FutureWarning:

The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead

c:\Users\admin\Projects\labs\5\3\tsc\practice\02 Similarity search\modules\utils.py:20: FutureWarning:

The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead



Выполните поиск похожих подпоследовательностей на запрос с помощью реализованных вами ранее алгоритмов MASS и UCR-DTW. Согласно истинной разметке, искомую активность человек выполнял 7 раз, поэтому параметр $topK=7$.

In [38]:
topK = 7
r = 0.1
excl_zone_frac = 1
is_normalize = True

# UCR-DTW
ucr_dtw_finder_opp = UCR_DTW(
    topK=topK,
    r=r,
    excl_zone_frac=excl_zone_frac,
    is_normalize=is_normalize
)
ucr_dtw_results_opp = ucr_dtw_finder_opp.perform(ts_opp, query_opp)

# MASS
dist_profile_mass_opp = mts.mass(ts_opp, query_opp)
excl_zone_opp = math.ceil(len(query_opp) * excl_zone_frac)
matches_mass_opp = topK_match(dist_profile_mass_opp, topK=topK, excl_zone=excl_zone_opp)

mass_bestmatch_results_opp = {
    "dist_profile": dist_profile_mass_opp,
    "matches": matches_mass_opp,
    "topK": topK,
    "excl_zone": excl_zone_opp,
}

Чтобы оценить качество распознавания активности, загрузите имеющуюся истинную разметку, хранящуюся в файле *labels.csv*. В разметке значением 1 обозначаются моменты времени, когда человек пил из чашки стоя, и значением 0 – в противном случае. Вычислите среди найденных $topK$ подпоследовательностей количество TP (True Positive) и FN (False Negative) с помощью функции `calculate_task1_accuracy()` и сравните между собой показатели, полученные с помощью MASS и UCR-DTW.

In [39]:
def calculate_task1_accuracy(labels: np.array, predicted_results: np.array) -> dict:
    """
    Calculate the accuracy of the algorithm which performs the activity recognition

    Parameters
    ----------
    labels: true labels
    predicted_results: results are predicted by algorithm

    Returns
    -------
        The number of True Positive and False Negative examples
    """

    TP = 0
    FN = 0

    topK = len(predicted_results['indices'])

    for i in range(topK):
        idx = predicted_results['indices'][i]
        if (labels[idx] == 1):
            TP = TP + 1

    FN = topK - TP

    return {'TP': TP,
            'FN': FN}

In [40]:
labels_opp_url = './datasets/part3/Opportunity/labels.csv'
labels_opp = read_ts(labels_opp_url).reshape(-1)

ucr_dtw_accuracy = calculate_task1_accuracy(labels_opp, ucr_dtw_results_opp['matches'])

mass_accuracy = calculate_task1_accuracy(labels_opp, mass_bestmatch_results_opp['matches'])

print(f"UCR-DTW Accuracy: {ucr_dtw_accuracy}")
print(f"MASS Accuracy: {mass_accuracy}")

UCR-DTW Accuracy: {'TP': 7, 'FN': 0}
MASS Accuracy: {'TP': 6, 'FN': 1}


c:\Users\admin\Projects\labs\5\3\tsc\practice\02 Similarity search\modules\utils.py:20: FutureWarning:

The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead



Итоговый вывод о преимуществе одной функции расстояния над другой в задаче поиска по образцу на данный момент делать еще рано, поэтому решим **вторую задачу из области транспорта**. Данная задача заключается в прогнозировании потока пешеходов в некоторой точке города на основе прошлых данных.

Прогнозирование ряда $T$ длины $n$ будем выполнять следующим образом. Сначала сформируем запрос $Q$, состоящий из $m$ последних по времени элементов ряда, $Q = T_{n-m+1,m}$. Далее среди всех подпоследовательностей ряда $T$, начинающихся с 1 до $n-2m$ позиций, найдем $topK$ похожих на запрос $Q$. Обозначим за $h > 0$ горизонт прогнозирования, определяющий количество элементов ряда, значения которых необходимо спрогнозировать. Для нахождения будущих значений будем брать $h$ элементов ряда, которые следуют за концевыми точками найденных похожих подпоследовательностей. В итоге, будущие значения будут получаться путем применения агрегатной функции к этим элементам. В качестве агрегатной функции может использоваться среднее арифметическое, взвешенное среднее, медиана и др.

Загрузите временной ряд из файла *pedestrian_count.csv*, расположенного в директории *./datasets/part3/Melbourne*. Данный ряд содержит почасовую статистику о количестве пешеходов на улице Бурке в Мельбурне (Австралия), являющейся одной из главных улиц города. Данные собраны за период с 1 марта 2020 по 31 октября 2022 гг. Полный набор данных и его детальное описание доступны по следующей [ссылке](https://data.melbourne.vic.gov.au/explore/dataset/pedestrian-counting-system-monthly-counts-per-hour/information/).

In [41]:
data_path = './datasets/part3/Melbourne/pedestrian_count.csv'

data = pd.read_csv(data_path, header=0)

Реализуйте алгоритм прогнозирования временного ряда на основе UCR-DTW и MASS, следуя приведенному выше описанию. За прогнозирование отвечает класс `BestMatchPredictor` из модуля *prediction.py*, дополните его недостающим кодом.  

Далее выполните прогнозирование потока людей для исходных данных на 1 день вперед (24 значения). Для этого установите следующие входные параметры:
* $h = 24$ (горизонт прогнозирования);
* $m = 168$ (длина запроса и подпоследовательностей, соответствующая 1 неделе);
* $aggr\_func = \text{'average'}$ (агрегатная функция);
* для алгоритма UCR-DTW:
    * $topK = 5$ (количество похожих подпоследовательностей на запрос);
    * $r = 0.1$ (ширина полосы Сако–Чиба);
    * $excl\_zone\_frac = 1$ (доля от длины подпоследовательностей, на основе которой определяется, являются ли подпоследоватлеьности тривиальными совпадениями);
    * $is\_normalize = True$ ($z$-нормализация запроса и подпоследовательностей);
* для алгоритма MASS:
    * $topK = 5$ (количество похожих подпоследовательностей на запрос);
    * $excl\_zone\_frac = 1$ (доля от длины подпоследовательностей, на основе которой определяется, являются ли подпоследоватлеьности тривиальными совпадениями).

Будем считать, что последние $h$ значений в загруженном временном ряде $T$ длины $n$ отсутствуют. Поэтому перед тем как находить прогнозные значения,  подготовьте данные. Разделите загруженный временной ряд $T$ на три части:
* ряд $T_{train}$, в котором будет выполняться поиск похожих подпоследовательностей на запрос $Q$: $T_{train} = T[0:(n-m-h)]$
* запрос $Q$: $Q = T[(n-m-h) : (n-h)]$
* реальные значения ряда $real\_values$, которые необходимо спрогнозировать: $real\_values = T[-h:] $

In [42]:
ucr_dtw_params = {
    'topK': 5,
    'r': 0.1,
    'excl_zone_frac': 1,
    'is_normalize': True
}

h = 24
m = 168
aggr_func = 'average'

ts_melbourne = data['Hourly_Counts'].values
n = len(ts_melbourne)

T_train = ts_melbourne[0:(n - m - h)]
Q = ts_melbourne[(n - m - h) : (n - h)]
real_values = ts_melbourne[-h:]

ucr_dtw_predictor = BestMatchPredictor(
    h=h,
    aggr_func=aggr_func,
    match_alg='UCR-DTW',
    match_alg_params=ucr_dtw_params
)

ucr_dtw_forecast = ucr_dtw_predictor.predict(T_train, Q)

print("UCR-DTW Forecast:", ucr_dtw_forecast)

UCR-DTW Forecast: [  5.  10.  18.  45.  87.  83. 138. 233. 394. 416. 452. 552. 749. 730.
 559. 481. 454. 444. 374. 246. 129.  32.  32.  15.]


In [43]:
mass_params = {
    'topK': 5,
    'excl_zone_frac': 1
}

mass_predictor = BestMatchPredictor(
    h=h,
    aggr_func=aggr_func,
    match_alg='MASS',
    match_alg_params=mass_params
)

mass_forecast = mass_predictor.predict(T_train, Q)

print("MASS Forecast:", mass_forecast)

MASS Forecast: [ 45.  29.  14.  14.   4.   5.   8.  48. 134. 254. 294. 357. 487. 808.
 830. 624. 592. 533. 571. 464. 307. 194. 150. 110.]


Далее выполните сравнение эффективности алгоритма при UCR-DTW и MASS по точности прогнозирования. Для оценки точности используйте **меру
среднеквадратичной ошибки RMSE (Root Mean Square Error)**, которая определяется следующим образом:
\begin{equation}
RMSE = \sqrt{\frac{1}{h}\sum_{i=1}^h{(t_i-\tilde{t}_i)}^2},
\end{equation}
где $t_i$ и $\tilde{t}_i$ — фактическое и прогнозное значения элемента ряда, $h$ — количество прогнозных элементов временного ряда.

In [44]:
from sklearn.metrics import mean_squared_error

rmse_ucr_dtw = np.sqrt(mean_squared_error(real_values, ucr_dtw_forecast))

rmse_mass = np.sqrt(mean_squared_error(real_values, mass_forecast))

print(f"RMSE for UCR-DTW based prediction: {rmse_ucr_dtw:.2f}")
print(f"RMSE for MASS based prediction: {rmse_mass:.2f}")

RMSE for UCR-DTW based prediction: 154.64
RMSE for MASS based prediction: 165.47


> Сделайте вывод о влиянии функции расстояния ED и DTW на точность решения задач интеллектульного анализа данных, которые основаны на алгоритмах поиска по образцу.  

Выбор между мерой DTW и ED для задач поиска по образцу зависит от характера временного ряда. Мера DTW демонстрирует значительное преимущество в задачах, где данные подвержены темпоральным деформациям, то есть когда форма искомого паттерна сохраняется, но его длительность или фаза могут варьироваться. Это подтверждается высокой точностью в задаче распознавания человеческой активности, где гибкость DTW в выравнивании последовательностей позволила успешно идентифицировать действия, выполненные с разной скоростью. Мера ED является более прагматичным выбором для анализа строго периодических и уже выровненных по времени данных, таких как прогнозирование пешеходного потока. В таких сценариях жесткое поэлементное сравнение обеспечивает сопоставимую точность, поскольку ключевые паттерны синхронизированы во времени. Учитывая, что вычислительная сложность ED (в реализации O(n log n)) на порядки ниже, чем у DTW (O(n·m·r)), его использование является более оправданным в задачах с большими объемами данных, где незначительный прирост точности от DTW не компенсирует существенные временные затраты.